In [1]:
import numpy as np
import mne
import io
import gzip
import matplotlib
import mplcursors 
import matplotlib.pyplot as plt
matplotlib.use('Qt5Agg')

In [2]:
raw = mne.io.read_raw_edf("P2_T1.edf", preload=True) 

Extracting EDF parameters from C:\Users\Suyash\Documents\Python\P2_T1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 55039  =      0.000 ...   214.996 secs...


In [3]:
emotiv_eeg_channels = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2','P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']

In [4]:
raw.pick(emotiv_eeg_channels)

<RawEDF | P2_T1.edf, 14 x 55040 (215.0 s), ~5.9 MiB, data loaded>

In [5]:
    full_montage = mne.channels.make_standard_montage("standard_1020")
    positions = full_montage.get_positions()["ch_pos"]
    selected_positions = {ch: pos for ch, pos in positions.items() if ch in emotiv_eeg_channels}

In [6]:
        custom_montage = mne.channels.make_dig_montage(ch_pos=selected_positions, coord_frame='head')
        raw.set_montage(custom_montage)

<RawEDF | P2_T1.edf, 14 x 55040 (215.0 s), ~5.9 MiB, data loaded>

In [7]:
raw.plot_sensors(show_names=True)

<Figure size 640x640 with 1 Axes>

In [8]:
raw.plot()

Using matplotlib as 2D backend.


<MNEBrowseFigure size 1920x1017 with 4 Axes>

In [9]:
raw.filter(l_freq=1, h_freq=None, fir_design='firwin')

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 845 samples (3.301 s)



<RawEDF | P2_T1.edf, 14 x 55040 (215.0 s), ~5.9 MiB, data loaded>

In [10]:
from mne.preprocessing import ICA

# Create an ICA object
ica_8 = ICA(n_components=8, random_state=97, max_iter=15000)

# Additional amplitude-based criteria
reject_criteria = dict(eeg=200e-6)  

# Fit the ICA model to raw the data
ica_8.fit(raw, reject=reject_criteria, flat=None, reject_by_annotation=True)


Fitting ICA to data using 14 channels (please be patient, this may take a while)
    Rejecting  epoch based on EEG : ['T7', 'AF4']
Artifact detected in [0, 512]
    Rejecting  epoch based on EEG : ['T8']
Artifact detected in [512, 1024]
    Rejecting  epoch based on EEG : ['P8']
Artifact detected in [2048, 2560]
    Rejecting  epoch based on EEG : ['P8']
Artifact detected in [2560, 3072]
    Rejecting  epoch based on EEG : ['P8', 'T8']
Artifact detected in [3072, 3584]
    Rejecting  epoch based on EEG : ['P8']
Artifact detected in [3584, 4096]
    Rejecting  epoch based on EEG : ['T7', 'AF4']
Artifact detected in [9216, 9728]
    Rejecting  epoch based on EEG : ['T8']
Artifact detected in [10240, 10752]
    Rejecting  epoch based on EEG : ['P8']
Artifact detected in [10752, 11264]
    Rejecting  epoch based on EEG : ['T8']
Artifact detected in [11264, 11776]
    Rejecting  epoch based on EEG : ['AF3', 'F7', 'F3', 'P7', 'FC6', 'AF4']
Artifact detected in [11776, 12288]
    Rejecting  e

Method,fastica
Fit parameters,algorithm=parallelfun=logcoshfun_args=Nonemax_iter=15000
Fit,17 iterations on raw data (38912 samples)
ICA components,8
Available PCA components,14
Channel types,eeg
ICA components marked for exclusion,—


In [11]:
sources = ica_8.get_sources(raw)

In [12]:
sources

<RawEDF | 8 x 55040 (215.0 s), ~3.4 MiB, data loaded>

In [13]:
psds = sources.compute_psd(fmin=1, fmax=50, n_fft=2048, picks="all")

Effective window size : 8.000 (s)


In [14]:
psds_db = 10 * np.log10(psds.get_data())

In [15]:
freqs = psds.freqs

In [16]:
psds_db.shape

(8, 393)

In [17]:
freqs.shape

(393,)

In [18]:
ica_8.plot_sources(raw)

Creating RawArray with float64 data, n_channels=8, n_times=55040
    Range : 0 ... 55039 =      0.000 ...   214.996 secs
Ready.


<MNEBrowseFigure size 1920x1017 with 4 Axes>

In [19]:
ica_8.plot_components()

Channels marked as bad:
none


<MNEFigure size 975x496 with 8 Axes>

In [20]:
sources = ica_8.get_sources(raw)
times = sources.times
data = sources.get_data().astype(np.float32)

In [21]:
data.shape

(8, 55040)

In [22]:
rawf_8 = raw.copy()

In [ ]:
### ------------Plotting individual ICA components Spectral Density -------------###

# Define frequency range
frequencies = np.logspace(0, 2, 100)

# Get a colormap
cmap = plt.get_cmap('plasma')

# Create a figure and axis
fig, ax = plt.subplots()

# Plot PSD of individual ICA components with distinct colors
for idx in range(ica_8.n_components_):
    color = cmap(idx / ica_8.n_components_)  # Use the colormap to get a distinct color
    psd, freqs = plt.psd(ica_8.get_sources(rawf_8).get_data()[idx], NFFT=1024, Fs=rawf_8.info['sfreq'], noverlap=512, label=f'Component {idx}', color=color)

# Add legend
plt.legend()

# Set plot labels
plt.title('Power Spectral Density of ICA Components')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power/Frequency (dB/Hz)')

# Zoom out after all components have been plotted
plt.autoscale()

# Enable hover tooltips
cursor = mplcursors.cursor(hover=True)

# Define the hover tooltip format
@cursor.connect("add")
def on_hover(sel):
    idx = int(sel.artist.get_label().split()[-1])  # Extract the component index from the label
    sel.annotation.set_text(f'Component {idx}')

# Display the plot
plt.show()

In [ ]:
muscle_idx_auto, scores = ica_8.find_bads_muscle(raw)
print(muscle_idx_auto)